# List of changes
Baseline
- Paper https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf
- Dropout
- Multi-head self-attention
- Character-level tokenization
- Decoder-only transformer with residual connections
- Feed forward with ReLU
- Layernorm
- AdamW optimizer, cross-entropy loss
- Absolute positional embeddings

V1
- SwiGLU instead of FFWD with ReLU
- No dropout
- RMSNorm instead of Layernorm

V2
- Weight decay
- Gradient clipping

V3
- scaled_dot_product_attention instead of my own self attention implementation
- Learnt tokenization, BPE (byte-pair encodings)

V4
- Smaller, 256 embed x 4 heads x 6 layers
- Scheduler (warm up plus cosine)

V5
- RoPE embeddings
- Bias true in attention layers
- KV cache
- AWP
- Merged multi-head attention

V6
- Paper https://arxiv.org/pdf/2305.07759
- 768 embed x 6 heads x 4 layers

<!-- V6
- Dropout back on
- Label smoothing -->

In [ ]:
# https://arxiv.org/pdf/2305.07759

In [ ]:
import torch
from torch import nn
from tqdm import tqdm
import matplotlib.pyplot as plt

from character_tokenizer import CharacterTokenizer

In [ ]:
BLOCK_SIZE = 1024
N_EMBED = 768 # Also called hidden size
N_HEADS = 6
N_LAYERS = 4

AMP = True  # Automatic Mixed Precision
TRAIN_SPLIT = 0.9
BATCH_SIZE = 64
LEARNING_RATE = 3e-4
MAX_ITERS = 40000
EVAL_ITERS = 20
EVAL_INTERVAL = 100

RANDOM_SEED = 1337

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if AMP: torch.set_float32_matmul_precision("high")

print("Hidden size:", N_EMBED//N_HEADS)
print("Tokens processed in training:", BATCH_SIZE * BLOCK_SIZE * MAX_ITERS / 1e6, "M")

In [ ]:
with open("datasets/chosen_dataset.info", "r", encoding="utf-8") as file:
    name = file.read().strip()
with open(f"datasets/{name}.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("Loaded dataset:", name, "with length", len(text)/1e6, "M characters")

In [ ]:
# load tokenizer
tokenizer = CharacterTokenizer()
tokenizer.load(f"intermediate/tokenizer")
ids = tokenizer.encode(text)

print("Vocab size:", tokenizer.vocab_size)
print("Tokens in training data:", len(ids)/1e6, "M, ( compression ratio:", len(ids)/len(text),")")
print("M, tokens processed in training:", BATCH_SIZE * BLOCK_SIZE * MAX_ITERS /1e6, "M ( number of epochs:", (BATCH_SIZE * BLOCK_SIZE * MAX_ITERS)/len(ids),")")

In [ ]:
data = torch.tensor(ids, dtype=torch.long)
print("Data tensor shape:", data.shape)
print("First 100 characters of the data tensor:", data[:100])

In [ ]:
n = int(TRAIN_SPLIT * len(data))
train_data = data[:n]
val_data = data[n:]

x = train_data[:BLOCK_SIZE]
y = train_data[1:BLOCK_SIZE + 1]
for t in range(BLOCK_SIZE):
    context = x[:t + 1]
    target = y[t]
    # print(f"Context: {context.tolist()} -> Target: {target.item()}")

In [ ]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i + 1:i + BLOCK_SIZE + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
# print("Inputs (x):", xb.shape, xb)
# print("Targets (y):", yb.shape, yb)

for b in range(BATCH_SIZE):
    for t in range(BLOCK_SIZE):
        context = xb[b, :t + 1]
        target = yb[b, t]
        # print(f"Batch {b}, Time {t}: Context: {context.tolist()} -> Target: {target.item()}")

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    """Rotary Position Embedding — applied to queries and keys in attention."""
    
    def __init__(self, dim, max_position_embeddings=BLOCK_SIZE, base=10000):
        super().__init__()
        self.dim = dim
        self.max_position_embeddings = max_position_embeddings
        self.base = base
        
        # θ_i = 10000^(-2i/dim) for i = 0, 1, ..., dim/2 - 1
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        
        self._set_cos_sin_cache(max_position_embeddings)
    
    def _set_cos_sin_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=torch.float32, device=self.inv_freq.device)
        freqs = torch.outer(t, self.inv_freq)          # (seq_len, dim/2)
        emb = torch.cat((freqs, freqs), dim=-1)         # (seq_len, dim)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)
    
    def forward(self, q, k, offset=0):
        # q, k: (B, num_heads, T, head_size)
        seq_len = q.size(2)  # ← was .size(1), now .size(2) because num_heads is dim 1
        total_len = offset + seq_len
        if total_len > self.max_position_embeddings:
            self._set_cos_sin_cache(total_len)
            self.max_position_embeddings = total_len
        
        cos = self.cos_cached[offset:offset + seq_len].to(q.dtype).unsqueeze(0).unsqueeze(0)  # (1, 1, T, head_size)
        sin = self.sin_cached[offset:offset + seq_len].to(q.dtype).unsqueeze(0).unsqueeze(0)  # (1, 1, T, head_size)
        
        q_embed = q * cos + self._rotate_half(q) * sin
        k_embed = k * cos + self._rotate_half(k) * sin
        return q_embed, k_embed
    
    @staticmethod
    def _rotate_half(x):
        x1 = x[..., :x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)

In [ ]:
# class HeadSelfAttention(nn.Module):
#     def __init__(self, head_size, bias=True):
#         super().__init__()
#         self.key = nn.Linear(N_EMBED, head_size, bias=bias)
#         self.query = nn.Linear(N_EMBED, head_size, bias=bias)
#         self.value = nn.Linear(N_EMBED, head_size, bias=bias)
#         self.RoPE = RotaryPositionalEmbedding(head_size)
#         # self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
#         # Register_buffer is used to store a tensor that is not a parameter but should be part of the module's state.

#     def forward(self, x, kv_cache=None, pos_offset=0):
#         B, T, C = x.shape
#         k = self.key(x)   # (B, T, head_size)
#         q = self.query(x) # (B, T, head_size)
#         v = self.value(x) # (B, T, head_size)

#         # Compute attention scores
#         # weights = q @ k.transpose(-2, -1) / torch.sqrt(C) # (B, T, T)
#         # weights = weights.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # Apply causal mask
#         # weights = torch.softmax(weights, dim=-1) # Normalize to get probabilities
#         # out = weights @ v # (B, T, head_size)

#         q, k = self.RoPE(q, k, offset=pos_offset)  # Apply Rotary Positional Embedding

#         if kv_cache is not None:
#             k_cache, v_cache = kv_cache
#             if k_cache is not None:
#                 k = torch.cat([k_cache, k], dim=1)  # Concatenate along the sequence dimension
#                 v = torch.cat([v_cache, v], dim=1)
#             kv_cache = (k, v)

#         is_causal = not (kv_cache is not None and kv_cache[0] is not None) # Cache exists -> We are at the end of a sequence -> nothing to mask out
#         out = nn.functional.scaled_dot_product_attention(q, k, v, is_causal=is_causal)
#         return out, kv_cache
    
# class MultiHeadSelfAttention(nn.Module):
#     def __init__(self, num_heads, head_size):
#         super().__init__()
#         self.heads = nn.ModuleList([HeadSelfAttention(head_size) for _ in range(num_heads)])
#         self.proj = nn.Linear(N_EMBED, N_EMBED) # Project concatenated heads back to embedding size

#     def forward(self, x, kv_cache=None, pos_offset=0):
#         head_outputs = []
#         new_cache = [] if kv_cache is not None else None
#         for i, head in enumerate(self.heads):
#             head_kv_cache = kv_cache[i] if kv_cache is not None else None
#             out, updated_kv_cache = head(x, kv_cache=head_kv_cache, pos_offset=pos_offset)
#             head_outputs.append(out)
#             if new_cache is not None:
#                 new_cache.append(updated_kv_cache)
#         out =  torch.cat(head_outputs, dim=-1) # Concatenate outputs from all heads
#         return self.proj(out), new_cache if new_cache is not None else None

class MultiHeadSelfAttention(nn.Module):
    """Merged multi-head attention — single batched QKV projection + single attention call."""
    
    def __init__(self, num_heads, head_size, bias=True):
        super().__init__()
        self.num_heads = num_heads
        self.head_size = head_size
        self.qkv = nn.Linear(N_EMBED, 3 * num_heads * head_size, bias=bias)
        self.proj = nn.Linear(num_heads * head_size, N_EMBED)
        self.RoPE = RotaryPositionalEmbedding(head_size)

    def forward(self, x, kv_cache=None, pos_offset=0):
        B, T, C = x.shape

        # Single projection to Q, K, V — all heads at once
        qkv = self.qkv(x)                                          # (B, T, 3 * N_EMBED)
        qkv = qkv.view(B, T, 3, self.num_heads, self.head_size)   # (B, T, 3, num_heads, head_size)
        qkv = qkv.permute(2, 0, 3, 1, 4)                          # (3, B, num_heads, T, head_size)
        q, k, v = qkv[0], qkv[1], qkv[2]                          # each (B, num_heads, T, head_size)

        # RoPE — works on last dim, leading dims (B, num_heads) broadcast fine
        q, k = self.RoPE(q, k, offset=pos_offset)

        # KV cache
        has_cache = kv_cache is not None and kv_cache[0] is not None
        if has_cache:
            k_cache, v_cache = kv_cache  # each (B, num_heads, cached_len, head_size)
            k = torch.cat([k_cache, k], dim=2)
            v = torch.cat([v_cache, v], dim=2)

        new_kv_cache = (k, v)

        # Single batched attention call
        is_causal = not has_cache
        out = nn.functional.scaled_dot_product_attention(q, k, v, is_causal=is_causal)
        # out: (B, num_heads, T, head_size)

        # Merge heads back
        out = out.transpose(1, 2).contiguous().view(B, T, -1)  # (B, T, num_heads * head_size)
        out = self.proj(out)
        return out, new_kv_cache

class SwiGLU(nn.Module):
    def __init__(self, n_embed, hidden_dim):
        super().__init__()
        self.w_gate = nn.Linear(n_embed, hidden_dim, bias=False)
        self.w_up   = nn.Linear(n_embed, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, n_embed, bias=False)
    def forward(self, x):
        return self.w_down(nn.functional.silu(self.w_gate(x)) * self.w_up(x))

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, n_embed, n_heads):
        super().__init__()
        head_size = n_embed // n_heads
        self.self_attention = MultiHeadSelfAttention(num_heads=n_heads, head_size=head_size)
        self.ffwd = SwiGLU(n_embed, 4 * n_embed)
        self.ln1 = nn.RMSNorm(n_embed, eps=1e-6)
        self.ln2 = nn.RMSNorm(n_embed, eps=1e-6)

    def forward(self, x, kv_cache=None, pos_offset=0):
        attn_out, new_kv_cache = self.self_attention(self.ln1(x), kv_cache=kv_cache, pos_offset=pos_offset)
        x = x + attn_out
        x = x + self.ffwd(self.ln2(x))
        return x, new_kv_cache

In [ ]:
class GPTLLM(nn.Module):
    def __init__(self, vocab_size, n_embed):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed) # Embedding for characters
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, n_embed) # Embedding for positions
        self.blocks = nn.ModuleList(
            [TransformerBlock(n_embed, N_HEADS) for _ in range(N_LAYERS)]
        )
        self.final_ln = nn.RMSNorm(n_embed, eps=1e-6)
        self.lm_head = nn.Linear(n_embed, vocab_size, bias=False) # Project embeddings to vocabulary

    def forward(self, idx, targets=None, kv_cache=None):
        pos_offset = 0
        if kv_cache is not None and kv_cache[0] is not None:
            pos_offset = kv_cache[0][0].size(2)  # Get the length of the cached keys for the first head

        token_embeds = self.token_embedding_table(idx) # (batch size, block size, n_embed)
        # pos_embeds = self.position_embedding_table(torch.arange(idx.size(1), device=idx.device)) # (block size, n_embed)
        # pos = torch.arange(pos_offset, pos_offset + idx.size(1), device=idx.device)
        # pos_embeds = self.position_embedding_table(pos)

        # x = token_embeds + pos_embeds # Combine token and position embeddings
        x = token_embeds # Use only token embeddings

        new_kv_cache = [] if kv_cache is not None else None
        for i, block in enumerate(self.blocks):
            block_kv = kv_cache[i] if kv_cache is not None else None
            x, block_kv_cache = block(x, kv_cache=block_kv, pos_offset=pos_offset)
            if new_kv_cache is not None:
                new_kv_cache.append(block_kv_cache)

        x = self.final_ln(x) # Apply final layer normalization
        logits = self.lm_head(x) # (batch size, block size, vocab size)
        
        if targets is None:
            return logits, None, new_kv_cache
        B, T, C = logits.shape
        logits = logits.view(B * T, C)
        targets = targets.view(B * T)

        loss = nn.functional.cross_entropy(logits, targets)#, label_smoothing=0.1)  # Apply label smoothing for better generalization

        return logits, loss, new_kv_cache
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        kv_cache = [None] * N_LAYERS  # Initialize key-value cache for each layer

        # prefill cache with the full prompt
        with torch.amp.autocast(device_type=device, enabled=AMP):
            logits, _, kv_cache = self(idx, kv_cache=kv_cache)
        # Sample first token directly from prefill output
        # (avoids re-feeding the last prompt token into the cache)
        logits = logits[:, -1, :]  # (B, vocab_size) — predicts token after prompt
        probabilities = nn.functional.softmax(logits / temperature, dim=-1)
        idx_next = torch.multinomial(probabilities, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)

        for _ in range(max_new_tokens):
            # idx_cond = idx[:, -BLOCK_SIZE:]  # crop context to the last BLOCK_SIZE tokens
            idx_cond = idx[:, -1:]
            # get the predictions
            with torch.amp.autocast(device_type=device, enabled=AMP):
                logits, loss, kv_cache = self(idx_cond, kv_cache=kv_cache)
            # focus on last time step
            logits = logits[:, -1, :]
            probabilities = nn.functional.softmax(logits / temperature, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probabilities, num_samples=1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

torch.manual_seed(RANDOM_SEED)
model = GPTLLM(vocab_size=tokenizer.vocab_size, n_embed=N_EMBED)
model = model.to(device)
logits, loss, _ = model(xb, yb)

print(loss)
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_sequence = model.generate(context, max_new_tokens=100)
print("Generated sequence:", tokenizer.decode(generated_sequence[0].tolist()))

In [ ]:
def count_parameters(model):
    """
    Returns a dict with total, trainable, non_trainable parameter counts
    and a per-top-module breakdown.
    """
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = total - trainable

    by_module = {}
    for name, p in model.named_parameters():
        top = name.split('.')[0]
        by_module[top] = by_module.get(top, 0) + p.numel()

    # sorted breakdown (module, count)
    breakdown = sorted(by_module.items(), key=lambda x: x[1], reverse=True)

    print(f"Total params: {total:,}")
    print(f"Trainable params: {trainable:,}")
    print(f"Non-trainable params: {non_trainable:,}")
    print("Top-level module breakdown:")
    for mod, cnt in breakdown:
        print(f"  {mod}: {cnt:,}")
count_parameters(model)

In [ ]:
@torch.no_grad()
def val_step():
    model.eval()
    losses = torch.zeros(EVAL_ITERS)
    for k in range(EVAL_ITERS):
        X, Y = get_batch('val')
        with torch.amp.autocast(device_type="cuda", enabled=AMP):
            logits, loss, _ = model(X, Y)
        losses[k] = loss.item()

    context = tokenizer.encode("KING ")
    context = torch.tensor(context, dtype=torch.long, device=device).unsqueeze(0)
    generated_sequence = model.generate(context, max_new_tokens=10)
    generated_text = tokenizer.decode(generated_sequence[0].tolist()).replace("\n", "[/n]")
    
    model.train()
    return losses.mean(), generated_text

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1)

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.01,
    total_iters=200
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_ITERS-200,
    eta_min=3e-5,   # about 10% of initial LR
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[200]
)

scaler = torch.amp.GradScaler(enabled=AMP)  # Initialize GradScaler for mixed precision training

history = {'train': [], 'val': [], 'lr': [], 'step': []}
pbar = tqdm(range(MAX_ITERS))
for step in pbar:
    xb, yb = get_batch('train')

    with torch.amp.autocast(device_type=device, enabled=AMP):  # Enable mixed precision for forward pass
        logits, loss, _ = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    
    # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping to prevent exploding gradients

    scaler.step(optimizer)
    scaler.update()
    
    scheduler.step()

    if step % EVAL_INTERVAL == 0:
        val_loss, generated_text = val_step()
        history['train'].append(loss.item())
        history['val'].append(val_loss)
        history['lr'].append(scheduler.get_last_lr()[0])
        history['step'].append(step)
        pbar.set_postfix({"Prompt response:": generated_text})
    pbar.set_description(f"Step {step}: Train Loss: {loss.item():.4f}, Val Loss: {val_loss:.4f}")

| Version              | Train loss | Train time [min] | Test loss | Test time [s] | Qualitative result |
|----------------------|:------------:|:------------------:|:-----------:|:---------------:|--------------------|
| (Andrej) Character tokenizer  | 1.195 |          45        |     1.490      |       18.6        |          Decent, some word errors, little sense though          |
| (Andrej) Learnt BPE tokenizer |     0.810       |      49            |    6.384       |     16.4          |         More sensical, less invented words           |
| (mine) Last version, shakespeare (1MB) |     0.062       |      16            |    7.013       |     5.1          |         Better           |
| (mine) Last version, tiny stories (22MB) |     1.229       |      17            |    2.046       |     5.3          |         Quite non-sensical aside from short sub-sentences          |

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(history['step'], history['train'], label='Train Loss')
ax1.plot(history['step'], history['val'], label='Validation Loss')
ax1.set_xlabel('Evaluation Step')
ax1.set_ylabel('Cross-Entropy loss')
ax1.set_yscale('log')

ax2 = ax1.twinx()
ax2.plot(history['step'], history['lr'], color='green', label='Learning Rate')
ax2.set_ylabel('Learning Rate')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

plt.title(f'Training Metrics - Train Loss: {history["train"][-1]:.4f}, Val Loss: {history["val"][-1]:.4f}')
plt.tight_layout()
plt.show()

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_sequence = model.generate(context, max_new_tokens=BLOCK_SIZE*2)
print("Generated sequence:", tokenizer.decode(generated_sequence[0].tolist()))

In [ ]:
prompt_list = [
    "Alice was so tired when she got back home so she went ",
    "Jack and Lily saw a rainbow after a rainy day. They were amazed by the colors. Jack said, 'Look, Lily. A rainbow has ",
    "Jack wanted to read a book, so he went to ",
    "'Can cows fly?', Alice asked her mother. "
]

for prompt in prompt_list:
    context = tokenizer.encode(prompt)
    context = torch.tensor(context, dtype=torch.long, device=device).unsqueeze(0)

    generated_sequence = model.generate(context, max_new_tokens=200)
    print("\n---------------\n", tokenizer.decode(generated_sequence[0].tolist()))

In [ ]:
# bag of words (averaging)
B, T, C = 4, 8, 2
torch.manual_seed(RANDOM_SEED)
x = torch.rand(B, T, C)
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1]
        xbow[b, t] = torch.mean(xprev, dim=0)

# second way
weights = torch.tril(torch.ones(T, T))
weights = weights / weights.sum(1, keepdim=True)
xbow2 = weights @ x

# third way
tril = torch.tril(torch.ones(T, T))
weights = torch.zeros((T, T))
weights = weights.masked_fill(tril == 0, float('-inf'))
weights = torch.nn.functional.softmax(weights, dim=-1)
xbow3 = weights @ x

# 4th, self attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
v = value(x)
weights = q @ k.transpose(-2, -1) * head_size ** -0.5

tril = torch.tril(torch.ones(T, T))
weights = weights.masked_fill(tril == 0, float('-inf'))
weights = torch.nn.functional.softmax(weights, dim=-1)
xbow4 = weights @ v